In [53]:
import pandas as pd
import numpy as np
import zipfile
import os
import glob
import warnings
warnings.filterwarnings("ignore")
from google.colab import files
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("quickcart_data")
print("Files extracted successfully:")
print(os.listdir("quickcart_data"))

Saving dim_events.zip to dim_events (1).zip
Files extracted successfully:
['dim_skus.csv', 'dim_suppliers.csv', 'dim_stores.csv', 'fact_inventory_daily.csv', 'dim_events.csv']


In [54]:
path = "quickcart_data"
stores = pd.read_csv(f"{path}/dim_stores.csv")
skus = pd.read_csv(f"{path}/dim_skus.csv")
suppliers = pd.read_csv(f"{path}/dim_suppliers.csv")
events = pd.read_csv(f"{path}/dim_events.csv")
inventory = pd.read_csv(f"{path}/fact_inventory_daily.csv")
print("Stores:", stores.shape)
print("SKUs:", skus.shape)
print("Suppliers:", suppliers.shape)
print("Events:", events.shape)
print("Inventory:", inventory.shape)

Stores: (12, 8)
SKUs: (60, 9)
Suppliers: (15, 6)
Events: (30, 5)
Inventory: (21600, 16)


In [55]:
target_counts = inventory["stockout_risk"].value_counts()
target_percent = inventory["stockout_risk"].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({ "Count": target_counts, "Percentage": target_percent.round(2)})
target_summary

,Count,Percentage
stockout_risk,,
Safe,14131,65.42
At-Risk,5186,24.01
Imminent,2283,10.57


In [56]:
inventory["date"] = pd.to_datetime(inventory["date"])
print("Minimum date:", inventory["date"].min().date())
print("Maximum date:", inventory["date"].max().date())
print("Unique dates:", inventory["date"].nunique())

Minimum date: 2026-10-01
Maximum date: 2026-10-30
Unique dates: 30


In [57]:
print(stores[["store_id", "city", "city_display"]])
stores["city_display"] = stores["city_display"].str.title()

   store_id       city city_display
0      ST01     Mumbai       Mumbai
1      ST02     Mumbai       Mumbai
2      ST03  Bengaluru    BENGALURU
3      ST04  Bengaluru    Bengaluru
4      ST05      Delhi        Delhi
5      ST06      Delhi        Delhi
6      ST07       Pune         Pune
7      ST08       Pune         Pune
8      ST09  Hyderabad    hyderabad
9      ST10  Hyderabad    Hyderabad
10     ST11    Chennai      Chennai
11     ST12    Chennai      Chennai


In [58]:
print("Missing reliability values:")
print(suppliers["reliability_score"].isna().sum())

Missing reliability values:
3


In [59]:
suppliers["reliability_score"] = pd.to_numeric( suppliers["reliability_score"], errors="coerce")
supplier_median = suppliers["reliability_score"].median()
suppliers["supplier_reliability_clean"] = suppliers["reliability_score"].fillna(supplier_median)
print("Median reliability:", supplier_median)
print("Remaining missing values:", suppliers["supplier_reliability_clean"].isna().sum())

Median reliability: 0.725
Remaining missing values: 0


In [60]:
df = inventory.copy()
df = df.merge( stores, on="store_id", how="left")
df = df.merge( skus, on=["sku_id", "supplier_id"], how="left", suffixes=("", "_sku"))
df = df.merge( suppliers, on="supplier_id", how="left", suffixes=("", "_supplier"))
events["date"] = pd.to_datetime(events["date"])
df = df.merge( events, on="date", how="left")
print("Final dataset shape:", df.shape)

Final dataset shape: (21600, 40)


In [61]:
print("Missing values after joins:")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
print(missing_values)

Missing values after joins:
lead_time_days_actual    19899
reliability_score         5040
dtype: int64


In [62]:
festival_effect = (
    df.groupby("is_festival_week")["stockout_risk"]
    .apply(lambda x: (x == "Imminent").mean() * 100)
)
festival_effect.round(2)

,stockout_risk
is_festival_week,
N,9.51
Y,23.31


In [63]:
perishable_effect = (
    df.groupby("is_perishable")["stockout_risk"]
    .apply(lambda x: (x == "Imminent").mean() * 100)
)
perishable_effect.round(2)

,stockout_risk
is_perishable,
N,9.30
Y,12.77


In [64]:
supplier_check = df.copy()
def reliability_group(x):
    if pd.isna(x):
        return "Missing"
    elif x < 0.75:
        return "Low"
    elif x < 0.85:
        return "Mid"
    else:
        return "High"
supplier_check["reliability_group"] = (
    supplier_check["reliability_score"]
    .apply(reliability_group)
)
supplier_effect = (
    supplier_check.groupby("reliability_group", dropna=False)["stockout_risk"]
    .apply(lambda x: (x == "Imminent").mean() * 100)
)
supplier_effect.round(2)

,stockout_risk
reliability_group,
High,3.82
Low,15.83
Mid,3.26
Missing,7.16


In [65]:
df = df.sort_values( ["store_id", "sku_id", "date"]).reset_index(drop=True)
df["reorder_gap"] = (df["reorder_point"] - df["closing_stock"])
df["days_of_cover_ratio"] = (
    df["days_of_cover"] /
    df["lead_time_days_expected"].replace(0, np.nan))
df["supplier_reliability_clean"] = (
    df["reliability_score"].fillna(
        df["reliability_score"].median()
    )
)
df["is_recent_reorder"] = (
    df.groupby(["store_id", "sku_id"])["reorder_placed"]
    .transform(
        lambda x:
        x.eq("Y")
        .shift(1)
        .rolling(3, min_periods=1)
        .max()
        .fillna(0)
        .astype(int)
    )
)
df["day_of_month"] = df["date"].dt.day
festival_start = pd.Timestamp("2026-10-22")
df["days_since_festival_start"] = np.where(
    df["date"] >= festival_start,
    (df["date"] - festival_start).dt.days,
    -1
)
df["city"] = df["city"].str.title()
print(df[
    [
        "reorder_gap",
        "days_of_cover_ratio",
        "supplier_reliability_clean",
        "is_recent_reorder",
        "day_of_month",
        "days_since_festival_start"
    ]
].head())

   reorder_gap  days_of_cover_ratio  supplier_reliability_clean  \
0       -108.5                 9.66                        0.83   
1        -98.5                10.79                        0.83   
2        -83.5                 8.99                        0.83   
3        -73.5                 8.79                        0.83   
4        -64.5                 8.55                        0.83   

   is_recent_reorder  day_of_month  days_since_festival_start  
0                  0             1                         -1  
1                  0             2                         -1  
2                  0             3                         -1  
3                  0             4                         -1  
4                  0             5                         -1  


In [66]:
category_risk = (
    df.groupby("category")["stockout_risk"]
    .apply(lambda x: (x == "Imminent").mean() * 100)
    .sort_values(ascending=False)
)
print(category_risk.round(2))

category
Bakery                 26.11
Beverages              14.55
Snacks                 10.19
Home Care               8.52
Fruits & Vegetables     7.93
Dairy                   7.54
Personal Care           6.79
Staples                 5.83
Name: stockout_risk, dtype: float64


In [67]:
train = df[ df["date"] <= "2026-10-23"].copy()
test = df[ df["date"] >= "2026-10-24"].copy()
print("Training rows:", len(train))
print("Testing rows:", len(test))

Training rows: 16560
Testing rows: 5040


In [68]:
target = "stockout_risk"
drop_columns = [ target, "date", "store_id", "sku_id", "supplier_id", "supplier_name", "sku_name", "categories_supplied", "city_tier", "lead_time_days_actual"]
X_train = train.drop(columns=drop_columns)
y_train = train[target]
X_test = test.drop(columns=drop_columns)
y_test = test[target]
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (16560, 35)
y_train: (16560,)
X_test: (5040, 35)
y_test: (5040,)


In [69]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
categorical_columns = X_train.select_dtypes( include="object").columns.tolist()
numerical_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_columns
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_columns
        )
    ]
)
print("Categorical features:", categorical_columns)
print("Numerical features:", len(numerical_columns))

Categorical features: ['reorder_placed', 'is_festival_week', 'city', 'city_display', 'store_size', 'category', 'is_perishable', 'festive_relevant', 'popularity_tier', 'event_name', 'event_type']
Numerical features: 24


In [70]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
baseline = DummyClassifier( strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print(classification_report(
    y_test,
    baseline_pred,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

     At-Risk     0.0000    0.0000    0.0000      1126
    Imminent     0.0000    0.0000    0.0000       775
        Safe     0.6228    1.0000    0.7676      3139

    accuracy                         0.6228      5040
   macro avg     0.2076    0.3333    0.2559      5040
weighted avg     0.3879    0.6228    0.4781      5040



In [71]:
from sklearn.linear_model import LogisticRegression
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])
logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
print(classification_report(
    y_test,
    logistic_pred,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

     At-Risk     0.9413    0.5986    0.7318      1126
    Imminent     0.7263    0.9484    0.8226       775
        Safe     0.9472    0.9994    0.9726      3139

    accuracy                         0.9020      5040
   macro avg     0.8716    0.8488    0.8423      5040
weighted avg     0.9119    0.9020    0.8957      5040



In [72]:
from sklearn.ensemble import RandomForestClassifier
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print(classification_report(
    y_test,
    rf_pred,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

     At-Risk     0.8138    0.9352    0.8702      1126
    Imminent     0.8893    0.6942    0.7797       775
        Safe     0.9981    0.9987    0.9984      3139

    accuracy                         0.9377      5040
   macro avg     0.9004    0.8760    0.8828      5040
weighted avg     0.9402    0.9377    0.9361      5040



In [73]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        random_state=42
    ))
])
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
print(classification_report(
    y_test,
    gb_pred,
    digits=4,
    zero_division=0
))

              precision    recall  f1-score   support

     At-Risk     0.8760    0.9414    0.9075      1126
    Imminent     0.9049    0.8103    0.8550       775
        Safe     1.0000    0.9990    0.9995      3139

    accuracy                         0.9571      5040
   macro avg     0.9270    0.9169    0.9207      5040
weighted avg     0.9577    0.9571    0.9567      5040



In [74]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
        0.6228,
        0.9016,
        0.9381,
        0.9575
    ],
    "Imminent Recall": [
        0.0000,
        0.9484,
        0.6942,
        0.8129
    ],
    "Imminent F1": [
        0.0000,
        0.8305,
        0.7820,
        0.8566
    ]
})
comparison

,Model,Accuracy,Imminent Recall,Imminent F1
0,Baseline,0.6228,0.0000,0.0000
1,Logistic Regression,0.9016,0.9484,0.8305
2,Random Forest,0.9381,0.6942,0.7820
3,Gradient Boosting,0.9575,0.8129,0.8566


In [75]:
gb = gb_model.named_steps["model"]
prep = gb_model.named_steps["preprocessor"]
feature_names = prep.get_feature_names_out()
importance = pd.Series( gb.feature_importances_, index=feature_names).sort_values(ascending=False)
print(importance.head(15))

num__days_of_cover                 0.411286
num__reorder_gap                   0.305490
num__days_of_cover_ratio           0.233016
num__supplier_reliability_clean    0.010095
num__reliability_score             0.008563
num__lead_time_variance_days       0.005281
cat__reorder_placed_Y              0.004158
num__closing_stock                 0.004132
cat__reorder_placed_N              0.003396
num__reorder_point                 0.003333
num__opening_stock                 0.002218
num__is_recent_reorder             0.001453
num__sales_velocity_7d             0.001316
num__day_of_month                  0.001235
num__unit_price_inr                0.001150
dtype: float64
